<a href="https://colab.research.google.com/github/RashiBista/GraphRAG/blob/main/02_knowledge_graph_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install requests

## Setting up Environment


In [2]:
import json
import re
from pathlib import Path
import requests
print("Environment Ready")

Environment Ready


## Loading chunks from Notebook 1

In [3]:
from pathlib import Path
import json

# Automatically mount Google Drive if a Drive path is requested
drive_path = Path("/content/drive/MyDrive/HybridGraphRAG/document_chunks.jsonl")
local_path = Path("/content/processed/document_chunks.jsonl")

# Check and mount if needed
if not drive_path.exists():
    try:
        from google.colab import drive
        print("Google Drive path not found. Attempting to mount Google Drive...")
        drive.mount('/content/drive')
    except ImportError:
        pass

# Determine which path to use
if drive_path.exists():
    chunks_path = drive_path
    print(f"Using Google Drive path: {chunks_path}")
elif local_path.exists():
    chunks_path = local_path
    print(f"Using local session path: {chunks_path}")
else:
    raise FileNotFoundError(
        f"Could not find the document_chunks.jsonl file at either:\n"
        f"- {drive_path}\n"
        f"- {local_path}\n"
        "Please ensure you have mounted your Google Drive or run Notebook 1 in this session."
    )

chunks = []
with chunks_path.open("r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

print("Number of chunks: ", len(chunks))

Using Google Drive path: /content/drive/MyDrive/HybridGraphRAG/document_chunks.jsonl
Number of chunks:  102


In [4]:
print(json.dumps(chunks[0], indent=2, ensure_ascii=False))

{
  "id": "12396518_p1_c0",
  "text": "Published online 23 August 2002\nBehavioural models of population growth rates:\nimplications for conservation and prediction\nWilliam J. Sutherland1* and Ken Norris2\n1Centre for Ecology, Evolution and Conservation, University of East Anglia, Norwich NR4 7TJ, UK\n2School of Animal & Microbial Sciences, University of Reading, Whiteknights, PO Box 228, Reading, RG6 6AJ, UK\nConservation biologists often wish to predict how vertebrate populations will respond to local or global\nchanges in conditions such as those resulting from sea-level rise, deforestation, exploitation, genetically\nmodiﬁed crops, global warming, human disturbance or from conservation activities. Predicting the conse-\nquences of such changes almost always requires understanding the population growth rate and the",
  "source": "12396518.pdf",
  "page": 1,
  "chunk_index": 0
}


## Extraction Pipeline

In [5]:
EXTRACTION_SCHEMA = {
    "entities": [
        {
            "name": "string",
            "type": "string"
        }
    ],
    "relationships": [
        {
            "source": "string",
            "relation": "string",
            "target": "string"
        }
    ]
}
print(json.dumps(EXTRACTION_SCHEMA, indent=2))

{
  "entities": [
    {
      "name": "string",
      "type": "string"
    }
  ],
  "relationships": [
    {
      "source": "string",
      "relation": "string",
      "target": "string"
    }
  ]
}


## Extraction Prompt

In [6]:
def build_extraction_prompt(text):
    return f"""
You are an information extraction system.

Extract important entities and relationships from the text below.

ENTITY TYPES may include:
- PERSON
- ORGANIZATION
- MODEL
- METHOD
- DATASET
- CONCEPT
- TECHNOLOGY
- TASK
- METRIC
- PAPER
- OTHER

For each relationship:
- source must be an entity name
- target must be an entity name
- relation should be a short uppercase phrase such as:
  USES
  BASED_ON
  DEVELOPED_BY
  EVALUATED_ON
  IMPROVES
  USED_FOR
  PART_OF
  COMPARED_WITH

Rules:
1. Only extract information explicitly supported by the text.
2. Do not invent entities.
3. Do not invent relationships.
4. Keep entity names concise.
5. Return valid JSON only.
6. Do not include markdown code fences.

Required format:

{{
  "entities": [
    {{
      "name": "Entity name",
      "type": "ENTITY_TYPE"
    }}
  ],
  "relationships": [
    {{
      "source": "Entity A",
      "relation": "RELATION",
      "target": "Entity B"
    }}
  ]
}}

TEXT:
{text}
"""

In [7]:
test_prompt = build_extraction_prompt(chunks[0]["text"])
print(test_prompt)


You are an information extraction system.

Extract important entities and relationships from the text below.

ENTITY TYPES may include:
- PERSON
- ORGANIZATION
- MODEL
- METHOD
- DATASET
- CONCEPT
- TECHNOLOGY
- TASK
- METRIC
- PAPER
- OTHER

For each relationship:
- source must be an entity name
- target must be an entity name
- relation should be a short uppercase phrase such as:
  USES
  BASED_ON
  DEVELOPED_BY
  EVALUATED_ON
  IMPROVES
  USED_FOR
  PART_OF
  COMPARED_WITH

Rules:
1. Only extract information explicitly supported by the text.
2. Do not invent entities.
3. Do not invent relationships.
4. Keep entity names concise.
5. Return valid JSON only.
6. Do not include markdown code fences.

Required format:

{
  "entities": [
    {
      "name": "Entity name",
      "type": "ENTITY_TYPE"
    }
  ],
  "relationships": [
    {
      "source": "Entity A",
      "relation": "RELATION",
      "target": "Entity B"
    }
  ]
}

TEXT:
Published online 23 August 2002
Behavioural models

## Mock extractor

In [8]:
def mock_extract(text):
  return{
      "entities":[],
      "relationships":[]
  }

In [9]:
result = mock_extract(chunks[0]["text"])
print(json.dumps(result, indent=2))

{
  "entities": [],
  "relationships": []
}


## Building a validator

In [10]:
from typing_extensions import Required
def validate_extraction(data):
  if not isinstance(data, dict):
    return False

  if "entities" not in data:
    return False

  if "relationships" not in data:
    return False

  if not isinstance(data["entities"], list):
    return False

  if not isinstance(data["relationships"], list):
    return False

  for entity in data["entities"]:
    if not isinstance(entity, data):
      return False

    if "name" not in entity or "type" not in entity:
      return False

  for relationship in data["relationships"]:
    if not isinstance(relationship, dict):
      return False

    required = {"source", "relation", "target"}

    if not required.issubset(relationship.keys()):
      return False

    if not required.issubset(relationship.keys()):
      return False

  return True



In [11]:
print(validate_extraction(result))

True


## Normalise

In [12]:
def normalize_entity_name(name):
    name = str(name).strip()
    name = re.sub(r"\s+", " ", name)
    return name

In [13]:
examples = [
    "   BERT  ",
    "BERT  model",
    "  Transformer   "
]

for item in examples:
    print(repr(normalize_entity_name(item)))

'BERT'
'BERT model'
'Transformer'


## A cleaner function

In [14]:
def normalize_extraction(data):
  entities = []
  relationships = []
  seen_entities = set()

  for entity in data.get("entities", []):
    name = normalize_entity_name(entity["name"])
    entity_type = str(entity["type"]).strip().upper()

    key = (name.lower(), entity_type)

    if key not in seen_entities:
      entities.append({
          "name": name,
          "type": entity_type
      })
      seen_entities.add(key)

    for rel in data.get("relationships", []):
      source = normalize_entity_name(rel["source"])
      relation = str(rel["relation"]).strip().upper()
      target = normalize_entity_name(rel["target"])

      if source and target and relation:
        relationships.append({
            "source": source,
            "relation": relation,
            "target": target
        })

  return {
      "entities": entities,
      "relationships": relationships
  }


## Creating a graph record for each count

In [15]:
def create_graph_record(chunk, extraction):
    return {
        "chunk_id": chunk["id"],
        "source": chunk["source"],
        "page": chunk["page"],
        "chunk_index": chunk["chunk_index"],
        "entities": extraction["entities"],
        "relationships": extraction["relationships"]
    }

In [16]:
normalized = normalize_extraction(result)

graph_record = create_graph_record(
    chunks[0], normalized
)
print(json.dumps(graph_record, indent=2))

{
  "chunk_id": "12396518_p1_c0",
  "source": "12396518.pdf",
  "page": 1,
  "chunk_index": 0,
  "entities": [],
  "relationships": []
}


## outpur dir

In [17]:
output_dir = Path("/content/processed")
output_dir.mkdir(parents=True, exist_ok=True)

In [18]:
test_records = [graph_record]

graph_output = output_dir / "graph_records.jsonl"

with graph_output.open("w", encoding="utf-8") as f:
    for record in test_records:
        f.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

print("Saved:", graph_output)

Saved: /content/processed/graph_records.jsonl


In [19]:
with graph_output.open("r", encoding="utf-8") as f:
    loaded_graph_records = [
        json.loads(line)
        for line in f
    ]

print(json.dumps(
    loaded_graph_records[0],
    indent=2,
    ensure_ascii=False
))

{
  "chunk_id": "12396518_p1_c0",
  "source": "12396518.pdf",
  "page": 1,
  "chunk_index": 0,
  "entities": [],
  "relationships": []
}


## Setting up Gemini SDK

In [20]:
!pip -q install -U google-genai

In [21]:
from google import genai
print("Gemini SDK ready")

Gemini SDK ready


In [22]:
from google.colab import userdata
# Make sure to add 'GEMINI_API_KEY' to your Colab Secrets (the Key icon on the left sidebar)
gemini_api_key = userdata.get('GEMINI_API_KEY')

In [23]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY was not found. "
        "Add it to Colab Secrets first."
    )

print("API key loaded successfully.")

API key loaded successfully.


In [24]:
client = genai.Client(api_key=GEMINI_API_KEY)
print("Gemini Key created successfully")

Gemini Key created successfully


In [25]:
EXTRACTION_SCHEMA = {
    "entities":[
        {
            "name":"string",
            "type":"string"
        }
    ],
    "relationships":[
        {
            "source":"string",
            "relation":"string",
            "target":"string"
        }
    ]
}

In [26]:
GEMINI_SCHEMA = {
    "type": "object",
    "properties": {
        "entities": {
            "type": "array",
            "description": "Important entities explicitly mentioned in the text.",
            "items": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "The concise name of the entity."
                    },
                    "type": {
                        "type": "string",
                        "description": "The entity category such as PERSON, ORGANIZATION, MODEL, METHOD, DATASET, CONCEPT, TECHNOLOGY, TASK, METRIC, PAPER, or OTHER."
                    }
                },
                "required": ["name", "type"]
            }
        },
        "relationships": {
            "type": "array",
            "description": "Relationships explicitly supported by the text.",
            "items": {
                "type": "object",
                "properties": {
                    "source": {
                        "type": "string",
                        "description": "Source entity name."
                    },
                    "relation": {
                        "type": "string",
                        "description": "Short uppercase relationship such as USES, BASED_ON, DEVELOPED_BY, EVALUATED_ON, IMPROVES, USED_FOR, PART_OF, or COMPARED_WITH."
                    },
                    "target": {
                        "type": "string",
                        "description": "Target entity name."
                    }
                },
                "required": ["source", "relation", "target"]
            }
        }
    },
    "required": ["entities", "relationships"]
}

In [27]:
import json
from google.genai import types

def extract_with_gemini(text):
    prompt = build_extraction_prompt(text)

    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=GEMINI_SCHEMA,
            temperature=0
        )
    )

    return json.loads(response.text)

In [28]:
test_chunk = chunks[0]

print("Chunk ID:", test_chunk["id"])
print("Source:", test_chunk["source"])
print("Page:", test_chunk["page"])
print("\nText:\n")
print(test_chunk["text"])

Chunk ID: 12396518_p1_c0
Source: 12396518.pdf
Page: 1

Text:

Published online 23 August 2002
Behavioural models of population growth rates:
implications for conservation and prediction
William J. Sutherland1* and Ken Norris2
1Centre for Ecology, Evolution and Conservation, University of East Anglia, Norwich NR4 7TJ, UK
2School of Animal & Microbial Sciences, University of Reading, Whiteknights, PO Box 228, Reading, RG6 6AJ, UK
Conservation biologists often wish to predict how vertebrate populations will respond to local or global
changes in conditions such as those resulting from sea-level rise, deforestation, exploitation, genetically
modiﬁed crops, global warming, human disturbance or from conservation activities. Predicting the conse-
quences of such changes almost always requires understanding the population growth rate and the


In [29]:
from google import genai
from google.genai import types
import json
from google.colab import userdata

# 1. Retrieve the fresh key from secrets
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    raise ValueError("GEMINI_API_KEY could not be found. Please add it to your Colab Secrets (key icon).")

# 2. Initialize the client
client = genai.Client(api_key=GEMINI_API_KEY)

# 3. Define the extraction helper function using gemini-3.8-flash
def extract_with_gemini(text):
    prompt = build_extraction_prompt(text)

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=GEMINI_SCHEMA,
            temperature=0
        )
    )
    return json.loads(response.text)

# 4. Run the extraction
try:
    result = extract_with_gemini(test_chunk["text"])
    print("✅ Extraction completed successfully!\n")
    print(json.dumps(result, indent=2))
except Exception as e:
    print(f"❌ Extraction failed: {e}")
    print("If you are seeing a 403 Permission Denied error, make sure your newly created API key is correctly saved in the Secrets tab (key icon) with 'Notebook access' enabled, and that you have restarted the runtime session to clear cached credentials.")

❌ Extraction failed: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your project has been denied access. Please contact support.', 'status': 'PERMISSION_DENIED'}}
If you are seeing a 403 Permission Denied error, make sure your newly created API key is correctly saved in the Secrets tab (key icon) with 'Notebook access' enabled, and that you have restarted the runtime session to clear cached credentials.


In [39]:
!pip -q install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.3 MB/s eta 0:00:00


In [40]:
from groq import Groq
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
if not GROQ_API_KEY:
  raise ValueError("GROQ_API_KEY was not found. Add it to Colab Secrets first.")

groq_client = Groq(api_key=GROQ_API_KEY)
print("API key loaded successfully")


API key loaded successfully


## connectivity Test


In [44]:
for m in groq_client.models.list().data:
    print(m.id)

whisper-large-v3-turbo
openai/gpt-oss-20b
canopylabs/orpheus-v1-english
canopylabs/orpheus-arabic-saudi
qwen/qwen3.8-27b
allam-2-7b
whisper-large-v3
openai/gpt-oss-safeguard-20b
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-prompt-guard-2-22m


In [45]:
MODEL = "openai/gpt-oss-safeguard-20b"

r = groq_client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with the word OK."}],
)
print(r.choices[0].message.content)

OK
